# PSF optimization

Import

In [ ]:
import sys
sys.path.insert(1, os.path.abspath('..'))  # repo root

import importlib
import SMLM ; importlib.reload(SMLM)
from SMLM import *


import numpy as np
import matplotlib.pyplot as plt
import pims
import os
from glob import glob
from dcimg import DCIMGFile
import scipy.optimize as opt
import time

Initialize classes camera and objective

In [ ]:
#camera and objective

# Orca Fusion cmos
cam = Camera(pixsize = 6500, # nm
             QE = 0.75, 
             baseline = 100, 
             amp = 0.24, 
             EM_gain = 1, 
             readnoise = 0.7)

# OLYMPUS APON 60X TIRF
obj = Objective(M = 60, 
                NA = 1.49,
                RI = 1.518,
                focal_len = 3e6,  # nm
                Ts_coefs =[-0.075, 0, -0.0694,  0, 1],
                Tp_coefs = [-0.0548, 0, 1])


In [ ]:
# load synthetic data test
path = r'C:\Users\q130jm\iCloudDrive\Documents\PhD\Coding\Python\PSF optimization'
filename = sorted(glob(f"{path}\\synthetic_data_try.tif"))
vid = [pims.open(n) for n in filename]; 
stack = np.array(vid[0])
stack = cts2phs(stack, cam, file_format='tif')
stack.shape

In [ ]:
from dcimg import DCIMGFile

# load dcimg file

pth = r'\\ces-1.i-med.ac.at\imed\Biomedizinische_Physik\groups\SMLM\STORM data\Nanorulers_80nm\Stack_2024_02_15'
file_name = '\\take_1_def_300.dcimg'

file = pth  + file_name
dcimg = DCIMGFile(file)
stack = np.array(cts2phs(dcimg[...], cam, file_format='dcimg'), dtype=np.uint16) # images [photons]

stack.shape
# Crop region from the image
x_start = 38
x_end = x_start + 25
y_start = 58
y_end = y_start + 25

cropped_image = stack[0][y_start:y_end, x_start:x_end]
print(np.min(stack[0]))
# Delete cropped region from the stack
stack[:, y_start:y_end, x_start:x_end] = np.min(stack[0])-1
plt.figure(figsize=(6,6))
plt.imshow(stack[0], cmap=parula)
plt.colorbar()
stack.shape



Load Stack

Preloc

In [ ]:
# preloc setup
image = stack[0]
sigma = 2               # sigma...sigma for Gaussian filter to smooth the raw images
min_dist = 7            # min_dist...minimum distance between molecules in pixels
threshold_abs = 24      # threshold_abs...minimum value of the cropped image to be accepted as molecule
threshold_rel = 2.5     # threshold_rel...threshold for peak detection: a molecule image rejected if max(image) < thresh * min(image)
N_crop=13               # pixel size of the cropped image
figsize = 7             # size of the figure
preloc_setup(image, sigma=sigma, min_dist=min_dist, threshold_abs=threshold_abs, threshold_rel=threshold_rel, N_crop=N_crop, figsize=figsize)



In [ ]:
prelocs, images = preloc_and_crop(stack, sigma=sigma, min_dist=min_dist, threshold_abs=threshold_abs, threshold_rel=threshold_rel, N_crop = N_crop)

In [ ]:
show_prelocs(images, n=24)

In [ ]:
# import aberrations

pth = r'\\ces-1.i-med.ac.at\imed\Biomedizinische_Physik\groups\SMLM\STORM data\Nanorulers_80nm\Stack_2024_02_15'
aberr = '\\aberr_stack.mat'
aberr_file = pth + aberr
aberr_file
noll_index, aberr, T_fit_coefs = load_aberr(aberr_file, mat_version = '7.3')

idx = np.where(np.isin(noll_index, [5, 6, 7, 8, 9, 10, 11, 22, 37, 56]))
noll_index = noll_index[idx]
aberr = aberr[idx]

noll_index, aberr

Initialize PSF

In [ ]:
# psf

#index = [5, 6, 7, 8, 9, 10, 11]
#aberr = [0, 0, 0, 0, 0, 0, 0]

h1 = psf(z_min = 0,
        z_max = 2000,
        z_defocus = 500,
        Nx = N_crop + 4, 
        ux = cam.pixsize/obj.M,
        uz = 50,
        Z_modes = noll_index,
        Z_magn = aberr,
        focal_len = obj.focal_len,
        Ts_coefs = obj.Ts_coefs,
        Tp_coefs = obj.Tp_coefs,
        T_fit_coefs = T_fit_coefs,
        os=2,
        correct_RI_mismatch = 'pure')

h2 = psf(z_min = 0,
        z_max = 2000,
        z_defocus = 1500,
        Nx = N_crop + 4, 
        ux = cam.pixsize/obj.M,
        uz = 50,
        Z_modes = noll_index,
        Z_magn = aberr,
        focal_len = obj.focal_len,
        Ts_coefs = obj.Ts_coefs,
        Tp_coefs = obj.Tp_coefs,
        T_fit_coefs = T_fit_coefs,
        os=2,
        correct_RI_mismatch = 'pure')


#h.show_slice_xy()       # plot the psf xy section
h1.show_projection()    # plot the psf z section
h2.show_projection()    # plot the psf z section

#h.show_pupil()
#h.show_CRLB(cam, s=1500, bg=40)

Fit

In [ ]:
# gpu fit
results, raw_results = perform_fit(images, prelocs, h, batch_size = 12000, show_progress = False)

In [ ]:
#show_results_3d(results)
show_results_2d(results, marker_size = 1)
show_results_1d(results, raw_results, marker_size = 1)
show_results_hist(results, raw_results, bins = 100)

In [ ]:
#FILTERING & VISUALIZATION
pxl = 1.5
filtered_results, filtered_raw_results, good_mol_idx = filter_results(results, raw_results,                                                        
            x_pxl_range=(-pxl, pxl),        # pixels
            y_pxl_range=(-pxl, pxl),        # pixels 
            z_range=(-50, 190),       # nm
            signal_range=(1, 6000),  # photons
            bg_range=(1, 200),      # photons
            LLR_max=120)
show_results_2d(filtered_results)
show_results_1d(filtered_results, filtered_raw_results, marker_size = 1)
show_results_hist(filtered_results, filtered_raw_results, bins = 60)

In [ ]:
h.RI

In [ ]:


aberr2 =  (h.aberr - (np.sum(h.aberr[h.pupil_UAF] * h.spher_def_sample[h.pupil_UAF]) / np.sum(h.spher_def_sample[h.pupil_UAF]**2)) * h.spher_def_sample)

vmin = np.min(np.real(h.aberr))
vmax = np.max(np.real(h.aberr))
cmap = 'hot'

plt.figure(figsize=(12,6))
plt.subplot(121)
plt.imshow(np.fft.fftshift(np.real(h.aberr)), vmin = vmin, vmax = vmax, cmap= cmap)
plt.colorbar()
plt.subplot(122)
plt.imshow(np.fft.fftshift(np.real(aberr2)), vmin = vmin, vmax = vmax, cmap = cmap)
plt.colorbar()



In [ ]:
import jax; import jax.numpy as jnp

def calc_image(a, v, M, z_def):
    """produces a molecule image given a set of zernikes and a parameter vector v=[x,y,z,sig,bg]"""

    aberr = jnp.fft.ifftshift(jnp.sum(h.modeset * a[:, jnp.newaxis, jnp.newaxis], axis = 0))

    aberr_pure =  (aberr - (np.sum(aberr[h.pupil_UAF] * h.D2[h.pupil_UAF]) / np.sum(h.D2[h.pupil_UAF]**2)) * h.D2)
    
    new_defocus,_ = h.calc_defocus( -z_def, h.RI[1])

    exp_jph = jnp.exp(1j*jnp.real(new_defocus + aberr_pure)) * h.apodization; # exponential of pupil phase.
    
    PSF = h.calc_molecule_image(h.z_vec, exp_jph) # calculate 3D psf

    I_hires = jax.image.scale_and_translate(PSF, 
                                            shape = (h.N_crop_os, h.N_crop_os, 1), #shape of images to be interpolated
                                            spatial_dims = (0, 1, 2), 
                                            scale = jnp.array([1, 1, 1]), 
                                            translation = jnp.array([v[0]-h.shift, v[1]-h.shift, -v[2]]), # y, x, z
                                            method = h.interp_method)[...,0] # tricubic interpolation gives pixelation results
   
   
    sig_scale = (M - M.min()).sum()
    if h.os > 1:
        I_hires = binning(I_hires, h.os)
        
    

    
    return v[4] + sig_scale * v[3] * I_hires

In [ ]:
@jax.jit
def fwd(p, M_v, z_def_v):
    
    a, v_v = p
    return h.chi2_mle(jax.vmap(lambda v, M, z: calc_image(a, v, M, z))(v_v, M_v, z_def_v), M_v)
    #return h.forward(a, v, images_good)

_grad =  jax.jit(jax.value_and_grad(fwd))

@jax.jit
def update(par, par_old, lrs, bounds, t1, M):

    err, par_bar = _grad(par, M)

    par = jax.tree_map(lambda p, lrs, g, bmin, bmax: jnp.clip(p - lrs * g, bmin,bmax), par, lrs, par_bar, bounds[0], bounds[1])
    t2 = 0.5 * (1 + jnp.sqrt(1 + 4 * t1**2))
    par = jax.tree_map(lambda p, po: p + (t1-1)/t2 * (p - po), par, par_old)
    par_old = jax.tree_map(lambda p: p, par)

    return err, par, par_old, t2

In [ ]:
sl = slice(1000)
#images_good = images[sl]
#prelocs_good = prelocs[sl]

images_good = images[good_mol_idx][sl]
prelocs_good = prelocs[good_mol_idx][sl]


In [ ]:

v_start = estimate_v0(images_good, h)
v_start = v_start.at[...,3].set(1)
aberr_start = jnp.zeros(len(noll_index),dtype=jnp.float32)


v_min = np.zeros_like(v_start)
v_max = np.zeros_like(v_start)

v_min[...,0] = -3. * h.os       # x
v_min[...,1] = -3. * h.os       # y 
v_min[...,2] = 0.               # z
v_min[...,3] = 1e-5             # sg
v_min[...,4] = 1.               # bg

v_max[...,0] = 3. * h.os        # x
v_max[...,1] = 3. * h.os        # y
v_max[...,2] = h.data.shape[2]  # z
v_max[...,3] = 1e5              # sg
v_max[...,4] = 200.             # bg

aberr_min = jnp.ones(len(noll_index)) * (-0.5)
aberr_max = jnp.ones(len(noll_index)) * (0.5)

bounds = ((aberr_min, jnp.array(v_min)),
          (aberr_max, jnp.array(v_max)))
         

ii = 10

plt.figure(figsize=(10,5))
plt.subplot(121)
plt.imshow(calc_image(aberr_start , v_start[ii], images_good[ii]), cmap=parula)
#plt.imshow(h.calc_molecule_image(h.z_vec, jnp.exp(1j * (h.modeset * aberr_start[:,None,None]).sum(axis=0)))[...,5])
plt.colorbar()

plt.subplot(122)
plt.imshow(images_good[ii], cmap=parula)
#plt.imshow(h.calc_molecule_image(h.z_vec, jnp.exp(1j * (h.modeset * aberr_start[:,None,None]).sum(axis=0)))[...,5])
plt.colorbar()


In [ ]:
maxiter = 1000
it = 0
#lrs = (1e-6, jnp.array([[1e-5,1e-5,1e-5,1e-5,1e-5]]))
lrs = (1e-7, jnp.ones((1, 5)) * 1e-4)
par = (aberr_start, v_start)
par_old = (jnp.zeros_like(aberr_start), jnp.zeros_like(v_start))
mom = 0.
n_img = len(images_good)
t1 = 1
while it < maxiter:

    err, par, par_old, t1 = update(par, par_old, lrs, bounds, t1, images_good) 
    
    print(it, err/n_img, np.round(par[0], 3))
    it += 1
aberr_rec, v_rec = par

In [ ]:
print(np.round([noll_index], 3))
print(np.round([aberr_rec], 3))
print(np.round([aberr], 3))

plt.plot(np.arange(len(aberr)), aberr, 'o', label='matlab')
plt.plot(np.arange(len(aberr)), aberr_rec, 'o', label='fit')
plt.xticks(np.arange(len(aberr)), noll_index)
plt.legend()
plt.show()

In [ ]:
ii = 10
plt.figure(figsize=(10,5))
plt.subplot(121)
plt.imshow(calc_image(aberr_rec , v_rec[ii], images_good[ii]), cmap=parula)
#plt.imshow(h.calc_molecule_image(h.z_vec, jnp.exp(1j * (h.modeset * aberr_start[:,None,None]).sum(axis=0)))[...,5])
plt.colorbar()

plt.subplot(122)
plt.imshow(images_good[ii], cmap=parula)
#plt.imshow(h.calc_molecule_image(h.z_vec, jnp.exp(1j * (h.modeset * aberr_start[:,None,None]).sum(axis=0)))[...,5])
plt.colorbar()

In [ ]:
LLR = np.array([h.chi2_mle(image, calc_image(aberr_rec, v_rec[i], image)) for i, image in enumerate(images_good)])
#LLR = np.zeros(len(images_good))

In [ ]:
v_rec_np = np.array(v_rec)
v_rec_np[:,3] *= (images_good - images_good.min()).sum(axis=(1,2))

results_simon, raw_results_simon = add_preloc_positions(v_rec_np, LLR, prelocs_good, h)
#show_results_3d(results_simon)
show_results_2d(results_simon, marker_size = 1)
show_results_1d(results_simon, raw_results_simon, marker_size = 1)
show_results_hist(results_simon, raw_results_simon, bins = 30)